In [1]:
# Importing csv file and load it into a working table for analysis
import numpy as np 
import pandas as pd 

import os
universal_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.csv'):
            universal_path = os.path.join(dirname, filename)
            break

if universal_path:
    print(f"Data successfully found at: {universal_path}")
    
    df = pd.read_csv(universal_path)
    print("Dataset loaded into dataframe 'df'. Ready for processing.")
else:
    print("Error: No CSV file found in the notebook data folder.")

Data successfully found at: /kaggle/input/datasets/bhumika1567/customerdata/Dataset.csv
Dataset loaded into dataframe 'df'. Ready for processing.


In [2]:
# Getting the insights from the table like number of columns, number of rows, column names, count of null values
print(df.shape)
print("---")
print(df.columns.tolist())
print("---")
print(df.dtypes)
print("---")
print(df.isnull().sum())
print("---")
print(df.head())

(3900, 18)
---
['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category', 'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season', 'Review Rating', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Payment Method', 'Frequency of Purchases']
---
Customer ID                 int64
Age                         int64
Gender                     object
Item Purchased             object
Category                   object
Purchase Amount (USD)       int64
Location                   object
Size                       object
Color                      object
Season                     object
Review Rating             float64
Subscription Status        object
Shipping Type              object
Discount Applied           object
Promo Code Used            object
Previous Purchases          int64
Payment Method             object
Frequency of Purchases     object
dtype: object
---
Customer ID                0
Age                        0
Gend

In [3]:
#duplicate values can cause interruption while analysing later so we take the count, as here count is zero so we dont need to solve this
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [96]:
#as earlier we saw that Review ratings has missing values we check how much missing it is of the total by taking percentage 
print("Missing Review Ratings:", df['Review Rating'].isnull().sum())
print("Total rows:", len(df))
print("Percentage missing:", round(df['Review Rating'].isnull().sum() / len(df) * 100, 2), "%")

Missing Review Ratings: 37
Total rows: 3900
Percentage missing: 0.95 %


In [4]:
#We observed that the percentage is minute so we fill the null values with median  
#We used median because it will not manipulate company's overall customer satisfaction as it takes the exact middle number

median_rating = df['Review Rating'].median()
print("Median Review Rating:", median_rating)

df['Review Rating'] = df['Review Rating'].fillna(median_rating)

print("Missing after filling:", df['Review Rating'].isnull().sum())

Median Review Rating: 3.8
Missing after filling: 0


In [5]:
text_columns = ['Gender', 'Category', 'Season', 'Subscription Status', 
                'Shipping Type', 'Discount Applied', 'Promo Code Used', 
                'Payment Method', 'Frequency of Purchases', 'Size']
#Putting all the columns and checking the unique values in those columns
#This is done to avoid the duplicate values , like different text can have same meaning (Male) , (male) , (M)
for col in text_columns:
    print(f"\n{col}:")
    print(df[col].unique())


Gender:
['Male' 'Female']

Category:
['Clothing' 'Footwear' 'Outerwear' 'Accessories']

Season:
['Winter' 'Spring' 'Summer' 'Fall']

Subscription Status:
['Yes' 'No']

Shipping Type:
['Express' 'Free Shipping' 'Next Day Air' 'Standard' '2-Day Shipping'
 'Store Pickup']

Discount Applied:
['Yes' 'No']

Promo Code Used:
['Yes' 'No']

Payment Method:
['Venmo' 'Cash' 'Credit Card' 'PayPal' 'Bank Transfer' 'Debit Card']

Frequency of Purchases:
['Fortnightly' 'Weekly' 'Annually' 'Quarterly' 'Bi-Weekly' 'Monthly'
 'Every 3 Months']

Size:
['L' 'S' 'M' 'XL']


In [99]:
#We observed 2 repeat words every 3 months and Quarterly 
# We observe there count and then merge them
print(df['Frequency of Purchases'].value_counts())

Frequency of Purchases
Every 3 Months    584
Annually          572
Quarterly         563
Monthly           553
Bi-Weekly         547
Fortnightly       542
Weekly            539
Name: count, dtype: int64


In [100]:
df['Frequency of Purchases'] = df['Frequency of Purchases'].replace('Every 3 Months', 'Quarterly')
print(df['Frequency of Purchases'].value_counts())

Frequency of Purchases
Quarterly      1147
Annually        572
Monthly         553
Bi-Weekly       547
Fortnightly     542
Weekly          539
Name: count, dtype: int64


In [101]:
# We changed customer ID datatype from int to object
df['Customer ID'] = df['Customer ID'].astype(str)
print("Customer ID type now:", df['Customer ID'].dtype)
print(df['Customer ID'].head()) 

Customer ID type now: object
0    1
1    2
2    3
3    4
4    5
Name: Customer ID, dtype: object


In [102]:
# Analysing each age is difficult hence we group them into age groups
df['Age Group'] = pd.cut(
    df['Age'], 
    bins=[0, 18, 25, 35, 50, 100], 
    labels=['Under 18', '18-25', '26-35', '36-50', '50+']
)

print(df['Age Group'].value_counts())

Age Group
50+         1476
36-50       1111
26-35        742
18-25        502
Under 18      69
Name: count, dtype: int64


In [103]:
# Categorising the buyers on basis of frequency of purchase
df['Purchase Frequency Segment'] = pd.cut(
    df['Previous Purchases'],
    bins=[0, 5, 15, 25, 100],
    labels=['New', 'Occasional', 'Regular', 'Loyal']
)

print(df['Purchase Frequency Segment'].value_counts())

Purchase Frequency Segment
Loyal         1935
Regular        786
Occasional     755
New            424
Name: count, dtype: int64


In [104]:
low = df['Purchase Amount (USD)'].quantile(0.33)
high = df['Purchase Amount (USD)'].quantile(0.66)

#Categorising based  on the purchase amount

print("Low threshold:", low)
print("High threshold:", high)

def segment_purchase(amount):
    if amount <= low:
        return 'Low'
    elif amount <= high:
        return 'Medium'
    else:
        return 'High'

df['Purchase Value Segment'] = df['Purchase Amount (USD)'].apply(segment_purchase)

print(df['Purchase Value Segment'].value_counts())

Low threshold: 45.0
High threshold: 73.0
Purchase Value Segment
Medium    1303
Low       1300
High      1297
Name: count, dtype: int64


In [105]:
#both discount applied and promo code are like offers and we wanted to check with offer and without offer purchase data
df['Any Discount'] = df.apply(
    lambda row: 'Yes' if row['Discount Applied'] == 'Yes' or row['Promo Code Used'] == 'Yes' else 'No',
    axis=1
)

print(df['Any Discount'].value_counts())

Any Discount
No     2223
Yes    1677
Name: count, dtype: int64


In [106]:
# calculating the loyalty score based on data
frequency_map = {'Weekly': 52, 'Bi-Weekly': 26, 'Fortnightly': 26, 'Monthly': 12, 'Quarterly': 4, 'Annually': 1}
df['Annual_Purchase_Frequency'] = df['Frequency of Purchases'].map(frequency_map)

df['Loyalty Score'] = (df['Annual_Purchase_Frequency'] + df['Previous Purchases']) * np.where(df['Subscription Status'] == 'Yes', 1.2, 1.0)
df['Loyalty Score'] = df['Loyalty Score'].round(1)

df['Is_Discount_Dependent'] = np.where((df['Any Discount'] == 'Yes'), True, False)
df['Est_Historical_Spent'] = df['Purchase Amount (USD)'] * df['Previous Purchases']
print(df['Loyalty Score'].describe())
print("\nSample:")
print(df[['Previous Purchases', 'Annual_Purchase_Frequency', 'Loyalty Score']].head(10))

count    3900.000000
mean       45.200923
std        23.978054
min         2.000000
25%        28.000000
50%        43.000000
75%        60.000000
max       122.400000
Name: Loyalty Score, dtype: float64

Sample:
   Previous Purchases  Annual_Purchase_Frequency  Loyalty Score
0                  14                         26           48.0
1                   2                         26           33.6
2                  23                         52           90.0
3                  49                         52          121.2
4                  31                          1           38.4
5                  14                         52           79.2
6                  49                          4           63.6
7                  19                         52           85.2
8                   8                          1           10.8
9                   4                          4            9.6


In [107]:
# Reviewing data once again
print("Final shape:", df.shape)
print("\nAll columns now:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)

Final shape: (3900, 26)

All columns now:
['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category', 'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season', 'Review Rating', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Payment Method', 'Frequency of Purchases', 'Age Group', 'Purchase Frequency Segment', 'Purchase Value Segment', 'Any Discount', 'Annual_Purchase_Frequency', 'Loyalty Score', 'Is_Discount_Dependent', 'Est_Historical_Spent']

Missing values:
Customer ID                   0
Age                           0
Gender                        0
Item Purchased                0
Category                      0
Purchase Amount (USD)         0
Location                      0
Size                          0
Color                         0
Season                        0
Review Rating                 0
Subscription Status           0
Shipping Type                 0
Discount Applied              0
Promo Code Used         

In [108]:
# Loading the final csv file 

df.to_csv('shopping_cleaned.csv', index=False)
print("File exported successfully")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

File exported successfully
Shape: (3900, 26)
Columns: ['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category', 'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season', 'Review Rating', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Payment Method', 'Frequency of Purchases', 'Age Group', 'Purchase Frequency Segment', 'Purchase Value Segment', 'Any Discount', 'Annual_Purchase_Frequency', 'Loyalty Score', 'Is_Discount_Dependent', 'Est_Historical_Spent']
